In [ ]:
# Bioinformatyka - 5 edycja
# Klastry obliczeniowe i rozwiązania chmurowe 2026/5_1
# pd5071

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("FASTQ_Analysis")
    .master("local[*]")
    .getOrCreate()
)

spark

In [4]:
# Ćwiczenie 7.1 Wczytanie surowych danych

file_path = "/home/clusters/work/SRR16356246_1_1.fastq"

lines_df = spark.read.text(file_path)

lines_df.show(8, truncate=False)

print(f"Liczba linii: {lines_df.count()}")
print(f"Liczba odczytów: {lines_df.count() // 4}")

# Liczba Jobs: 5. 2 Jobs zawierały dodatkowo Skipped Stage, wynikający z optymalizacji planu wykonania.
 
# Stages: Każdy Job składał się z 1 Stage.
# Każdy Stage wykonał 1 Task (`1/1`), ponieważ plik wejściowy został przetworzony jako jedna partycja.

# Cache: Nie zastosowano (cache() ani persist() nie zostały użyte).

# Locality Level: PROCESS_LOCAL (dane były przetwarzane lokalnie w jednym kontenerze Docker).

# Wnioski: akcje `show()` i `count()` uruchamiają osobne Jobs. 
# Ponieważ wykonano kilka akcji na tym samym DataFrame bez użycia cache, Spark ponownie odczytywał dane przy każdej akcji.
# Wczytanie pliku utworzyło jedynie plan wykonania, natomiast Jobs zostały uruchomione dopiero przez akcje show() i count().

+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                  |
+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|@SRR16356246.1 1 length=151                                                                                                                            |
|TTAGAGAGGGAGTGAAGNGAATGTTGCTGAGGTTTTCCAGCACTGTGACATATGGCCANTTCTGTTTTCNTGTAGCAAAACCAGAAATCCTGACTTACGACAGGCTCGTGAATGGCATGCTCCAATGTGTGGCAGCAGGATTCCCAGAGCC|
|+SRR16356246.1 1 length=151                                                                                                                            |
|???????????????????????????????????????????????????????????????????????????

In [5]:
# Ćwiczenie 7.2.1 Dodanie numerów lini

lines_with_id = lines_df.withColumn(
    "line_number",
    monotonically_increasing_id()
)
lines_with_id.show(8, truncate=False)

# Liczba Jobs: 1

# Stages: 1.

# Cache: Nie wykorzystano.

# Locality Level: PROCESS_LOCAL.

# Wnioski: Dodanie kolumny za pomocą monotonically_increasing_id() jest transformacją leniwą.
# Job został uruchomiony dopiero przez akcję show().
# Operacja nie wymagała shuffle ani dodatkowych etapów wykonania.

+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|value                                                                                                                                                  |line_number|
+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|@SRR16356246.1 1 length=151                                                                                                                            |0          |
|TTAGAGAGGGAGTGAAGNGAATGTTGCTGAGGTTTTCCAGCACTGTGACATATGGCCANTTCTGTTTTCNTGTAGCAAAACCAGAAATCCTGACTTACGACAGGCTCGTGAATGGCATGCTCCAATGTGTGGCAGCAGGATTCCCAGAGCC|1          |
|+SRR16356246.1 1 length=151                                                                                                                            |2          |
|???

In [6]:
# Ćwiczenie 7.2.2 Dodanie numerów lini

from pyspark.sql.window import Window

window = Window.orderBy(monotonically_increasing_id())

lines_with_id = lines_df.withColumn(
    "line_number",
    row_number().over(window) - 1
)

lines_with_id.show(8, truncate=False)

# Liczba Jobs: 1

# Stages: 1.

# Cache: Nie wykorzystano.

# Locality Level: PROCESS_LOCAL.

# Wnioski: Numeracja z wykorzystaniem funkcji row_number() i Window została wykonana
# podczas akcji show(). W planie wykonania pojawił się operator Window,
# co oznacza dodatkowe przetwarzanie związane z funkcją okna,
# jednak dla danych lokalnych operacja nadal została wykonana w jednym Stage.

+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|value                                                                                                                                                  |line_number|
+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|@SRR16356246.1 1 length=151                                                                                                                            |0          |
|TTAGAGAGGGAGTGAAGNGAATGTTGCTGAGGTTTTCCAGCACTGTGACATATGGCCANTTCTGTTTTCNTGTAGCAAAACCAGAAATCCTGACTTACGACAGGCTCGTGAATGGCATGCTCCAATGTGTGGCAGCAGGATTCCCAGAGCC|1          |
|+SRR16356246.1 1 length=151                                                                                                                            |2          |
|???

In [7]:
# Ćwiczenie 7.3 Określenie typu linii w FASTQ

lines_typed = lines_with_id.withColumn(
    "line_type",
    when(col("line_number") % 4 == 0, "header")
    .when(col("line_number") % 4 == 1, "sequence")
    .when(col("line_number") % 4 == 2, "separator")
    .when(col("line_number") % 4 == 3, "quality")
)

lines_typed.show(12, truncate=False)

# Liczba Jobs: 2

# Stages: po 1 Stage.

# Cache: Nie.

# Locality Level: PROCESS_LOCAL.

# Wnioski: Dodanie kolumny line_type za pomocą withColumn() i when() jest transformacją leniwą.
# Obliczenia zostały wykonane dopiero podczas akcji show().
# W planie wykonania pojawił się operator Exchange (shuffle), wynikający z wcześniejszego
# wykorzystania numeracji wierszy z funkcją Window.

+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+
|value                                                                                                                                                  |line_number|line_type|
+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+
|@SRR16356246.1 1 length=151                                                                                                                            |0          |header   |
|TTAGAGAGGGAGTGAAGNGAATGTTGCTGAGGTTTTCCAGCACTGTGACATATGGCCANTTCTGTTTTCNTGTAGCAAAACCAGAAATCCTGACTTACGACAGGCTCGTGAATGGCATGCTCCAATGTGTGGCAGCAGGATTCCCAGAGCC|1          |sequence |
|+SRR16356246.1 1 length=151                                                                                            

In [8]:
# Ćwiczenie 7.4 Utworzenie identyfikatora odczytu

lines_with_record = lines_typed.withColumn(
    "record_id",
    (col("line_number") / 4).cast("integer")
)

lines_with_record.show(16, truncate=False)

# Liczba Jobs: 2

# Stages: 1 Stage. Po 1 task.

# Cache: Nie.

# Locality Level: PROCESS_LOCAL.

# Shuffle: Nie wystąpił. Widoczny jest jedynie zapis Shuffle Write (13.2 KiB), 
# wynikający z wcześniejszego planu wykonania (np. zastosowania funkcji okna), 
# natomiast sama operacja dodania kolumny record_id nie powoduje wymiany danych między partycjami.

# Wnioski: withColumn() dodające kolumnę record_id jest transformacją leniwą (lazy transformation) i nie uruchamia obliczeń samodzielnie.
# Wykonanie nastąpiło dopiero po wywołaniu akcji show().
# Obliczenie record_id polega wyłącznie na prostym wyrażeniu arytmetycznym wykonywanym dla każdego wiersza 
# (line_number / 4 oraz rzutowanie na integer), dlatego nie wymagało przetasowania danych (shuffle) ani zmiany partycjonowania.
# Ponieważ line_number został wcześniej wyznaczony z użyciem funkcji okna (row_number()), Spark wykorzystał istniejący plan wykonania. 
# Sama operacja dodania record_id nie zwiększyła złożoności obliczeń.


+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+---------+
|value                                                                                                                                                  |line_number|line_type|record_id|
+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+---------+
|@SRR16356246.1 1 length=151                                                                                                                            |0          |header   |0        |
|TTAGAGAGGGAGTGAAGNGAATGTTGCTGAGGTTTTCCAGCACTGTGACATATGGCCANTTCTGTTTTCNTGTAGCAAAACCAGAAATCCTGACTTACGACAGGCTCGTGAATGGCATGCTCCAATGTGTGGCAGCAGGATTCCCAGAGCC|1          |sequence |0        |
|+SRR16356246.1 1 length=151                                          

In [9]:
# Ćwiczenie 7.5 Przekształcenie do formatu szerokiego (Pivot)

fastq_wide = lines_with_record.groupBy("record_id") \
    .pivot("line_type") \
    .agg(first("value"))

fastq_wide.show(5, truncate=False)

fastq_wide.printSchema()

# Liczba Jobs: 4 ( 1 - przygotowanie operacji pivot() (wyznaczenie wartości kolumn line_type),
#                  2 – właściwe wykonanie show(). Skipped Stage,
#                  3 – wykonanie printSchema(),
#                  4 – dodatkowy etap agregacji związany z operacją pivot().Skipped Stage.)

# Stages: po 1 Stage dla każdego Job.
 
# Cache: Nie zastosowano.

# Locality Level: PROCESS_LOCAL.

# Wnioski: pivot() jest najbardziej kosztowną operacją spośród dotychczas wykonanych transformacji.
# W planie wykonania pojawia się operator Exchange, który oznacza wykonanie shuffle, czyli redystrybucję danych pomiędzy partycjami przed grupowaniem.
# Po wykonaniu shuffle Spark realizuje agregację za pomocą operatorów SortAggregate, grupując rekordy według record_id.
# Widoczne są również operatory WholeStageCodegen, które świadczą o zastosowaniu optymalizacji polegającej na 
# wygenerowaniu zoptymalizowanego kodu Java dla całego fragmentu planu wykonania.
# Dodatkowe Joby wynikają z faktu, że pivot() przed właściwą agregacją musi ustalić zestaw unikalnych wartości kolumny line_type, 
# a następnie wykonać właściwe grupowanie i agregację.

+---------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|record_id|header                     |quality                                                                                                                                                |separator                  |sequence                                                                                                                                               |
+---------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------+-------------------------------------

In [10]:
# Ćwiczenie 7.6 Czyszczenie danych nagłówka

fastq_clean = fastq_wide.withColumn(
    "read_id",
    split(
        regexp_replace(col("header"), "^@", ""),
        " "
    )[0]
)

fastq_clean.select(
    "record_id",
    "read_id",
    "sequence",
    "quality"
).show(5, truncate=False)

# Liczba Jobs: 2 (1 - przygotowanie planu wykonania dla transformacji oraz rozpoczęcie przetwarzania.
#                 2 - wykonanie akcji show(), wyświetlenie wyników.)

# Stages: 1 Stage. W ostatnim Job zastosowano Skipped Stage, ponieważ Spark ponownie wykorzystał fragment wcześniej przygotowanego planu wykonania.

# Cache: Nie.

# Locality Level: PROCESS_LOCAL.

# Wnioski: Operacje regexp_replace() oraz split() są transformacjami kolumnowymi wykonywanymi lokalnie na każdym rekordzie.
# Same funkcje nie wymagają shuffle, ponieważ nie powodują grupowania ani redystrybucji danych pomiędzy partycjami.
# W planie wykonania nadal widoczny jest operator Exchange, ale nie został on wywołany przez regexp_replace() ani split(). Jest to element odziedziczony z wcześniejszej operacji pivot(), od której rozpoczyna się przetwarzanie DataFrame fastq_wide.
# Widoczne są również operatory SortAggregate oraz WholeStageCodegen, ponieważ Spark wykonuje cały plan pochodzący od wcześniejszych transformacji i optymalizuje go jako jeden ciąg operacji.
# Transformacja withColumn("read_id", ...) jest leniwa (lazy) i została wykonana dopiero podczas wywołania akcji show().
# Operacje regexp_replace() i split() nie wprowadzają dodatkowego shuffle. 
# Widoczny w planie operator Exchange pochodzi z wcześniejszej operacji pivot(), a nie z czyszczenia nagłówka.

+---------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|record_id|read_id      |sequence                                                                                                                                               |quality                                                                                                                                                |
+---------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|0        